# 11 - Model Analysis
**Fake News Detection using Machine Learning and NLP**

## Overview
Comparative analysis of all trained SIGMA models.

## Step 1 - Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score
)

sns.set_style('whitegrid')
print('Libraries imported successfully!')

## Step 2 - Load Preprocessed Dataset

In [ ]:
df = pd.read_csv('../dataset/processed_fake_news_dataset.csv')
text_col = 'processed_text'
if text_col not in df.columns:
    text_col = 'content' if 'content' in df.columns else 'text'
df = df.dropna(subset=[text_col, 'label'])

tfidf = TfidfVectorizer(max_features=5000, max_df=0.7, min_df=2)
X = tfidf.fit_transform(df[text_col])
y = df['label'].values
feature_names = tfidf.get_feature_names_out()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'X_test: {X_test.shape}')

## Step 3 - Load Saved Models

In [ ]:
model_names = {
    'SVM': 'sigma_svm.pkl',
    'Decision Tree': 'sigma_decision_tree.pkl',
    'Naive Bayes': 'sigma_naive_bayes.pkl',
    'AdaBoost': 'sigma_adaboost.pkl',
    'Gradient Boosting': 'sigma_gradient_boosting.pkl'
}

models = {}
for name, file in model_names.items():
    path = os.path.join('../sigma_module/outputs', file)
    if os.path.exists(path):
        models[name] = joblib.load(path)
    else:
        print(f'[WARN] {path} not found.')
print(f'Loaded {len(models)} models.')

## Step 4 - Generate Predictions and Metrics

In [ ]:
results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_prob = model.decision_function(X_test)
    else:
        y_prob = y_pred.astype(float)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results[name] = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'y_prob': y_prob
    }
    print(f'{name}: Acc={acc:.4f}')

## Step 5 - Visualizations
### 5.1 ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison - SIGMA Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 Performance Bar Chart

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
keys = ['accuracy', 'precision', 'recall', 'f1']
x = np.arange(len(results))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
for i, key in enumerate(keys):
    values = [results[n][key] for n in results.keys()]
    ax.bar(x + i*width, values, width, label=metrics[i])

ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(results.keys())
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
ax.set_title('Model Performance Comparison')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()